In [1]:
import os
import re
import requests
import subprocess
from pathlib import Path
from tqdm import tqdm


def get_direct_file_link(mailru_file_url: str) -> str:
    """
    Преобразует публичную ссылку вида:
        https://cloud.mail.ru/public/<key>/<subkey>/<filename>
    в прямую ссылку на CDN, по которой можно скачать файл через wget или requests.

    Возвращает прямую ссылку для скачивания.
    """
    resp = requests.get(mailru_file_url)
    if resp.status_code != 200:
        raise RuntimeError(f"Ошибка {resp.status_code} при запросе {mailru_file_url}")

    match = re.search(r'dispatcher.*?weblink_get.*?url":"(.*?)"', resp.text)
    if not match:
        raise RuntimeError("Не удалось найти CDN ссылку в HTML Mail.ru")

    base_url = match.group(1)
    parts = mailru_file_url.strip("/").split("/")[-3:]
    return f"{base_url}/{parts[0]}/{parts[1]}/{parts[2]}"


def download_from_mailru(file_url: str, local_name: str, force: bool = False, show_progress: bool = True):
    """
    Скачивает файл с Mail.ru по публичной ссылке.

    Args:
        file_url: ссылка на файл в облаке Mail.ru.
        local_name: имя файла для сохранения.
        force: если True — перекачивает даже если файл уже есть.
        show_progress: показывать ли прогресс-бар.
    """
    local_path = Path(local_name)
    if local_path.exists() and not force:
        print(f"Файл {local_name} уже существует, пропускаем скачивание.")
        return

    direct = get_direct_file_link(file_url)
    print(f"Скачиваем {file_url} → {local_name}")

    with requests.get(direct, stream=True) as r:
        r.raise_for_status()
        total_size = int(r.headers.get("content-length", 0))
        block_size = 8192
        with open(local_name, "wb") as f, tqdm(
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=f"Downloading {local_name}",
            disable=not show_progress,
        ) as bar:
            for chunk in r.iter_content(block_size):
                f.write(chunk)
                bar.update(len(chunk))

    print(f"Файл {local_name} успешно скачан ({os.path.getsize(local_name)/1e6:.1f} MB).")

In [2]:
# Ссылки на данные по задаче
train_link = "https://cloud.mail.ru/public/Gsyr/8VxmbhAaZ/train_data.tar"
test_link  = "https://cloud.mail.ru/public/Gsyr/8VxmbhAaZ/test_data.tar"

In [3]:
# Если скорость загрузки низкая — это может быть связано с CDN.
# Попробуйте перезапустить ячейку: при новом соединении может попасться другой узел CDN,
# и загрузка обычно проходит быстрее (2-3 минуты при нормальном узле).
download_from_mailru(train_link, "train_data.tar")
download_from_mailru(test_link, "test_data.tar")

Скачиваем https://cloud.mail.ru/public/Gsyr/8VxmbhAaZ/train_data.tar → train_data.tar


Файл train_data.tar успешно скачан (2472.0 MB).
Скачиваем https://cloud.mail.ru/public/Gsyr/8VxmbhAaZ/test_data.tar → test_data.tar


Файл test_data.tar успешно скачан (741.0 MB).


In [4]:
!tar -xvf train_data.tar &> logs.txt
!tar -xvf test_data.tar &> logs.txt

In [5]:
!pip install faster-whisper -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 45.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.4 MB

In [6]:
import os
import pandas as pd
from faster_whisper import WhisperModel
import torch
from tqdm.auto import tqdm

# --- КОНФИГУРАЦИЯ ---
TEST_AUDIO_DIR = "/kaggle/working/test_opus/audio"
SUBMISSION_FILE = "vseros-audio.csv"

# "large-v3" — текущая SOTA модель whisper.
# В faster-whisper можно использовать "large-v2" или просто "large", если v3 работает хуже на специфичных данных.
MODEL_SIZE = "large-v3"

# Определяем устройство и тип вычислений
if torch.cuda.is_available():
    DEVICE = "cuda"
    # float16 значительно ускоряет работу на GPU без потери качества
    COMPUTE_TYPE = "float16"
else:
    DEVICE = "cpu"
    # int8 - стандартная квантованная версия для CPU
    COMPUTE_TYPE = "int8"

print(f"Используется устройство: {DEVICE}, тип вычислений: {COMPUTE_TYPE}")

TARGET_PHRASES = ["не слышу", "не слышно"]

# --- ФУНКЦИИ ---

def normalize_text(text):
    """Нормализация текста для поиска фраз."""
    if not text:
        return ""
    text = text.lower()
    for char in ".,!?":
        text = text.replace(char, "")
    return text.strip()

def contains_target_phrase(text):
    """Проверка на наличие целевых фраз."""
    normalized_text = normalize_text(text)
    for phrase in TARGET_PHRASES:
        if phrase in normalized_text:
            return True
    return False

# --- ЗАГРУЗКА МОДЕЛИ ---

print(f"Загрузка модели Faster-Whisper: '{MODEL_SIZE}'...")
# Модель загружается один раз с параметрами оптимизации
model = WhisperModel(
    MODEL_SIZE,
    device=DEVICE,
    compute_type=COMPUTE_TYPE
)
print("Модель успешно загружена.")

# --- ПОДГОТОВКА ФАЙЛОВ ---

if not os.path.exists(TEST_AUDIO_DIR):
    print(f"Создаю тестовую директорию для проверки (так как путь {TEST_AUDIO_DIR} не найден)...")
    os.makedirs(TEST_AUDIO_DIR, exist_ok=True)

test_files = [
    f for f in os.listdir(TEST_AUDIO_DIR)
    if f.endswith('.opus') and not f.startswith('._')
]

# (Опционально) Если файлов нет, код не упадет, а просто выведет сообщение
if not test_files:
    print(f"В папке {TEST_AUDIO_DIR} не найдены аудиофайлы (.opus).")
else:
    print(f"Найдено {len(test_files)} тестовых файлов для обработки.")

results = []

# --- ОБРАБОТКА ---

for audio_file in tqdm(test_files, desc="Обработка аудиофайлов"):
    audio_path = os.path.join(TEST_AUDIO_DIR, audio_file)
    audio_id = os.path.splitext(audio_file)[0]

    try:
        # В faster-whisper метод transcribe возвращает генератор сегментов, а не словарь
        segments, info = model.transcribe(
            audio_path,
            language="ru",
            beam_size=5, # beam_size=5 дает лучшее качество, beam_size=1 — максимальную скорость
            vad_filter=True # Встроенный VAD фильтр, убирает тишину (ускоряет процессинг)
        )

        # Собираем текст из сегментов
        # Генератор запускается именно в этот момент (ленивое вычисление)
        text_segments = [segment.text for segment in segments]
        full_text = " ".join(text_segments).strip()

        label = 1 if contains_target_phrase(full_text) else 0

    except Exception as e:
        print(f"Ошибка при обработке файла {audio_file}: {e}")
        full_text = ""
        label = 0

    results.append({"id": audio_id, "label": label, "text": full_text})

# --- СОХРАНЕНИЕ ---

submission_df = pd.DataFrame(results)
submission_df.to_csv(SUBMISSION_FILE, index=False)

print(f"Обработка завершена. Результаты сохранены в файл: {SUBMISSION_FILE}")
if not submission_df.empty:
    print(f"Пример предсказаний:\n{submission_df.head()}")
    print(f"\nСтатистика по предсказанным меткам:\n{submission_df['label'].value_counts(normalize=True)}")

Используется устройство: cuda
Загрузка модели Whisper: 'large'...


100%|██████████████████████████████████████| 2.88G/2.88G [00:26<00:00, 115MiB/s]


Модель успешно загружена.
Найдено 27000 тестовых файлов для обработки.


Обработка аудиофайлов:   0%|          | 0/27000 [00:00<?, ?it/s]

Обработка завершена. Результаты сохранены в файл: vseros-audio.csv
Пример предсказаний:
                                         id  label  \
0  7234618762472991490500645211265109235107      0   
1  2319653480327446792415555403743643275694      1   
2  4312281317231076130209891127097377948427      0   
3  1221135949234494615107074368114284175784      1   
4  2933045844027770635132565038463141219712      1   

                                                text  
0   Я никому из знакомых девушек из сферы не слыш...  
1                               Не видно, не слышно.  
2             Вот чему меня жизнь когда-то наказала.  
3         Отличная. Вообще не слышно уличных звуков.  
4                       Не слышно даже всхлипываний.  

Статистика по предсказанным меткам:
label
0    0.530259
1    0.469741
Name: proportion, dtype: float64
